In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
# import arviz as az
# import bambi as bmb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import spearmanr, pearsonr, ttest_1samp
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages


In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import statistics_utils

In [ ]:
from membandit.plotting_style import set_plot_style


set_plot_style()

## Load the behavioral data: 

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)


# First, let's drop reaction times that are physiologically too fast
learn_df = learn_df[learn_df.gamble_rt>0.3]
drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3], 
                          combined_df[combined_df.recog_rt<0.3]])
combined_df.drop(index=drop_trials.index, inplace=True)

# Next, let's drop reaction times that are slower than the 97.5th percentile of each subject's reaction times
for subj in learn_df.participant.unique():
    rt_threshold = np.percentile(learn_df[learn_df.participant==subj].gamble_rt, 95)
    learn_df.loc[(learn_df.participant==subj) & (learn_df.gamble_rt>rt_threshold), 'gamble_rt'] = np.nan
    combined_df.loc[(combined_df.participant==subj) & (combined_df.gamble_rt_gamble>rt_threshold), 'gamble_rt_gamble'] = np.nan
    learn_df.loc[(learn_df.participant==subj), 'rpe_cat'] = pd.qcut(learn_df.loc[(learn_df.participant==subj), 'rpe'], 3, labels=['neg', 'neutral', 'pos'])
    combined_df.loc[(combined_df.participant==subj), 'rpe_cat'] = pd.qcut(combined_df.loc[(combined_df.participant==subj), 'rpe'], 3, labels=['neg', 'neutral', 'pos'])



learn_df.dropna(subset=['gamble_rt'], inplace=True)
combined_df.dropna(subset=['gamble_rt_gamble'], inplace=True)

# 3/18/2025

# recover the latent Q-values
combined_df['Q'] = combined_df.rpe + combined_df.reward_gamble
# compute the prior trial's rpe:
combined_df['prior_rpe'] = combined_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
combined_df['trials_since_reversal'] = combined_df.apply(lambda x: np.mod(x.trials_gamble, reversal_trials[int(x.block_gamble-2)]) if x.block_gamble>1 else x.trials_gamble, axis=1)
# recode hits as 0.5 and misses as -0.5
combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5 if x.hits==False else np.nan, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward_gamble==1 else -0.5 if x.reward_gamble==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
combined_df['pgamble'] = combined_df.apply(lambda x: 0.5 if x.gamble_gamble==1 else -0.5 if x.gamble_gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in combined_df.participant.unique():

        combined_df.loc[combined_df.participant==subj, 
                'zQ'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'Q']))
        combined_df.loc[combined_df.participant==subj,
                'zprior_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        combined_df.loc[combined_df.participant==subj, 
                        'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                        'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'DPRIME']))

        combined_df.loc[combined_df.participant==subj,
                'ztrials_since_reversal'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((combined_df.loc[combined_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        combined_df.loc[combined_df.participant==subj,
                'RT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'reward_gamble'] - RT_inferred_EV_subj)

        combined_df.loc[combined_df.participant==subj,
                'zRT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'RT_rpe']))

In [ ]:
# What is the correlation between reward and rpe categories?
sns.barplot(data=learn_df, y='reward', x='rpe_cat')

## Load the neural data

In [ ]:
# LOAD the data
analysis_evs = ['feedback_start']
chan_dfs = [] 
glms = []
elec_counter = 0 
block = [0.0, 1.5]
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')

            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            # chan_df['rpe_cat'] = pd.qcut(chan_df.rpe, 3, labels=['neg', 'neutral', 'pos'])
            chan_df['urpe'] = np.abs(chan_df.rpe)
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
            chan_df['preward'] = chan_df.apply(lambda x: 0.5 if x.reward==1 else -0.5 if x.reward==0 else np.nan, axis=1)

            # # # learning only: 
            chan_df = chan_df[chan_df.gamble_rt>0.3]
            chan_df = chan_df[chan_df.gamble_rt<10]

            # # # with memory
            # memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits', 'recog_rt', 'zrpe', 'zQ', 'zRT_rpe', 'preward', 'ztrials_since_reversal', 'zprior_rpe']]
            # memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
            # memory_df['phit'] = memory_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)
            # chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])

            # chan_df = chan_df[chan_df.recog_rt>0.3]
            # chan_df = chan_df[chan_df.recog_rt<10]

            elec_counter +=1

            chan_dfs.append(chan_df)

chan_dfs_feedback = pd.concat(chan_dfs)
chan_dfs_feedback['zAge'] =(chan_dfs_feedback.age - np.nanmean(chan_dfs_feedback.age)) / (2*np.nanstd(chan_dfs_feedback.age))

rois = ['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC', 'dlPFC']
# remove rows where region is not in the list   
chan_dfs_feedback = chan_dfs_feedback[chan_dfs_feedback.region.isin(rois)]

# Electrode level permutation regression

## Electrode Regression Analysis Function

This notebook provides a comprehensive function for performing regression analysis between TFR (time-frequency representation) and behavioral variables for each electrode, with permutation-based statistical testing.

### Key Features:

1. **Per-electrode analysis**: Runs regression separately for each unique electrode
2. **Frequency band filtering**: Analyze specific frequency bands (alpha, beta, hfa, etc.)
3. **Permutation testing**: Z-scores beta weights against shuffled data distributions
4. **Multiple comparison correction**: FDR correction for p-values
5. **Flexible variables**: Use any combination of independent variables
6. **Comprehensive output**: Beta coefficients, z-scores, p-values, and R²

### Function Parameters:

- `data`: The chan_dfs_feedback dataframe
- `fband`: Frequency band to analyze ('alpha', 'beta', 'hfa', 'high_theta', 'low_theta')
- `independent_vars`: List of variables (e.g., ['zrpe', 'zQ', 'zpm'])
- `n_permutations`: Number of permutations for z-scoring (default: 1000)
- `min_trials`: Minimum trials per electrode (default: 10)

### Available Variables:

Based on the data structure, you can use variables like:
- `zrpe`: Z-scored reward prediction error
- `zQ`: Z-scored Q-value
- `zpm`: Z-scored memory performance (d')
- `zRT_rpe`: Z-scored reaction time-based RPE
- `preward`: Coded reward (-0.5/0.5)
- `pgamble`: Coded choice (-0.5/0.5)
- `ztrials_since_reversal`: Z-scored trials since reversal
- `zprior_rpe`: Z-scored prior trial RPE
- `zAge`: Z-scored age

### Output Columns:

For each electrode and variable, the function returns:
- `{variable}_beta`: Raw beta coefficient
- `{variable}_z_score`: Z-score against permutation distribution
- `{variable}_p_value`: Uncorrected p-value
- `{variable}_p_fdr`: FDR-corrected p-value
- `actual_r2`: Model R²
- `r2_z_score`: R² z-score against permutation distribution
- `r2_p_value`: R² p-value

### Usage Examples:

```python
# Simple RPE analysis
results = electrode_regression_analysis(chan_dfs_feedback, 'alpha', ['zrpe'])

# Multiple variables
results = electrode_regression_analysis(chan_dfs_feedback, 'beta', ['zrpe', 'zQ', 'zpm'])

```

In [ ]:
def electrode_regression_analysis(data, fband, independent_vars, n_permutations=1000, 
                                 random_state=42, min_trials=10):
    """
    Perform regression analysis between TFR and independent variables for each electrode,
    with permutation-based z-scoring of beta weights.
    
    Parameters:
    -----------
    data : pandas.DataFrame
        The chan_dfs_feedback dataframe containing TFR and behavioral data
    fband : str
        Frequency band to analyze (e.g., 'alpha', 'beta', 'hfa', 'high_theta', 'low_theta')
    independent_vars : list
        List of independent variable names to include in regression
    n_permutations : int, default=1000
        Number of permutations for z-scoring
    random_state : int, default=42
        Random seed for reproducibility
    min_trials : int, default=10
        Minimum number of trials required per electrode for analysis
        
    Returns:
    --------
    results_df : pandas.DataFrame
        DataFrame with regression results for each electrode
    """
    
    import numpy as np
    import pandas as pd
    from scipy import stats
    import statsmodels.formula.api as smf
    from sklearn.linear_model import LinearRegression
    from sklearn.preprocessing import StandardScaler
    import warnings
    warnings.filterwarnings('ignore')
    
    # Set random seed
    np.random.seed(random_state)
    
    # Filter data for specific frequency band
    data = data[data['fband'] == fband]
    
    if len(data) == 0:
        raise ValueError(f"No data found for frequency band '{fband}'")
    
    # Initialize results storage
    results = []
    
    # Get unique electrodes
    unique_electrodes = data['unique_elec_id'].unique()
    
    print(f"Analyzing {len(unique_electrodes)} electrodes for frequency band '{fband}'")
    
    for i, elec_id in enumerate(unique_electrodes):
        if i % 50 == 0:  # Progress indicator
            print(f"Processing electrode {i+1}/{len(unique_electrodes)}")
            
        # Get data for this electrode
        elec_data = data[data['unique_elec_id'] == elec_id].copy()
        
        # Check minimum trial requirement
        if len(elec_data) < min_trials:
            continue
            
        # Prepare variables for regression
        y = elec_data['tfr'].values
        
        # Check for valid independent variables
        valid_vars = []
        X_data = []
        
        for var in independent_vars:
            if var in elec_data.columns:
                var_data = elec_data[var].values
                # Check for sufficient non-NaN values
                if np.sum(~np.isnan(var_data)) >= min_trials:
                    valid_vars.append(var)
                    X_data.append(var_data)
        
        if len(valid_vars) == 0:
            continue
            
        # Stack independent variables
        X = np.column_stack(X_data)
        
        # Remove rows with any NaN values
        valid_mask = ~(np.isnan(y) | np.any(np.isnan(X), axis=1))
        y_clean = y[valid_mask]
        X_clean = X[valid_mask]
        
        if len(y_clean) < min_trials:
            continue
            
        # Perform actual regression
        try:
            reg = LinearRegression().fit(X_clean, y_clean)
            actual_betas = reg.coef_
            actual_r2 = reg.score(X_clean, y_clean)
            
            # Permutation testing
            permuted_betas = []
            permuted_r2s = []
            
            for perm in range(n_permutations):
                # Shuffle the dependent variable (TFR)
                y_shuffled = np.random.permutation(y_clean)
                
                # Fit regression on shuffled data
                reg_perm = LinearRegression().fit(X_clean, y_shuffled)
                permuted_betas.append(reg_perm.coef_)
                permuted_r2s.append(reg_perm.score(X_clean, y_shuffled))
            
            # Convert to arrays for easier computation
            permuted_betas = np.array(permuted_betas)
            permuted_r2s = np.array(permuted_r2s)
            
            # Compute z-scores for each beta coefficient
            z_scores = []
            p_values = []
            
            for j, var in enumerate(valid_vars):
                perm_mean = np.mean(permuted_betas[:, j])
                perm_std = np.std(permuted_betas[:, j])
                
                if perm_std > 0:
                    z_score = (actual_betas[j] - perm_mean) / perm_std
                    # Two-tailed p-value
                    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
                else:
                    z_score = 0
                    p_value = 1
                
                z_scores.append(z_score)
                p_values.append(p_value)
            
            # Z-score for R²
            r2_z_score = (actual_r2 - np.mean(permuted_r2s)) / np.std(permuted_r2s) if np.std(permuted_r2s) > 0 else 0
            r2_p_value = 2 * (1 - stats.norm.cdf(abs(r2_z_score))) if np.std(permuted_r2s) > 0 else 1
            
            # Store results
            electrode_info = elec_data.iloc[0]  # Get electrode metadata
            
            result_row = {
                'unique_elec_id': elec_id,
                'participant': electrode_info['participant'],
                'region': electrode_info['region'],
                'channel': electrode_info['channel'],
                'hemi': electrode_info['hemi'],
                'fband': fband,
                'n_trials': len(y_clean),
                'actual_r2': actual_r2,
                'r2_z_score': r2_z_score,
                'r2_p_value': r2_p_value,
            }
            
            # Add results for each independent variable
            for j, var in enumerate(valid_vars):
                result_row.update({
                    f'{var}_beta': actual_betas[j],
                    f'{var}_z_score': z_scores[j],
                    f'{var}_p_value': p_values[j],
                })
            
            results.append(result_row)
            
        except Exception as e:
            print(f"Error processing electrode {elec_id}: {e}")
            continue
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # Add FDR correction for multiple comparisons
    if len(results_df) > 0:
        from statsmodels.stats.multitest import multipletests
        
        # Collect all p-values for FDR correction
        p_value_cols = [col for col in results_df.columns if col.endswith('_p_value')]
        
        for col in p_value_cols:
            valid_p_values = results_df[col].dropna()
            if len(valid_p_values) > 0:
                _, p_corrected, _, _ = multipletests(valid_p_values, method='fdr_bh')
                results_df[f"{col.replace('_p_value', '_p_fdr')}"] = np.nan
                results_df.loc[valid_p_values.index, f"{col.replace('_p_value', '_p_fdr')}"] = p_corrected
    
    print(f"Analysis complete. Results for {len(results_df)} electrodes.")
    return results_df


In [ ]:
# Example usage of the regression function

# Define frequency bands available in the data
available_bands = chan_dfs_feedback['fband'].unique()
print("Available frequency bands:", available_bands)

# Example 1: Simple regression with RPE
print("\n=== Example 1: TFR ~ RPE (z-scored) ===")
results_rpe = electrode_regression_analysis(
    data=chan_dfs_feedback,
    fband='hfa',  # Choose frequency band
    independent_vars=['reward'],  # RPE z-scored
    n_permutations=500,  # Fewer permutations for faster testing
    min_trials=15
)

print(f"Results shape: {results_rpe.shape}")
print(f"Columns: {list(results_rpe.columns)}")
print(f"\nTop 5 electrodes by R² z-score:")
# print(results_rpe.nlargest(5, 'r2_z_score')[['region', 'channel', 'actual_r2', 'r2_z_score', 'zrpe_beta', 'zrpe_z_score']])


In [ ]:
results_rpe['abs_reward_beta'] = np.abs(results_rpe['reward_beta'])

In [ ]:
# save out results_rpe with the name of the pertinent predictor 
results_rpe.to_csv(f'{base_dir}/scratch/MemoryBandit/results/electrode_regress_reward.csv', index=False)


In [ ]:
results_rpe.groupby(['region']).mean(numeric_only=True).sort_values(by='abs_reward_beta', ascending=False)

In [ ]:
# IS there a correlation between the abs_reward_beta coding in any region and RL parameters?

# 1. merge results_rpe with learn_df on 'participant'
merged_df = results_rpe.merge(learn_df, on='participant', how='left')

# 2. run a mixedlm with the following formula: 'abs_reward_beta ~ 1 + alpha'
mixedlm = smf.mixedlm('reward_z_score ~ 1 + alpha', 
groups='participant',
data=merged_df[merged_df.region.isin(['HPC'])])

# 3. run the model
model = mixedlm.fit()

# 4. print the summary
print(model.summary())



In [ ]:
# plot the relationship between abs_reward_beta and alpha for each region
sns.regplot(data=merged_df[merged_df.region.isin(['dlPFC'])], x='alpha', y='abs_reward_beta', scatter=True)

10/21/2025: So this is nice - dmPFC, dlPFC, OFC, ACC all code reward at a higher level than HPC and AMY - a good validation to start. 

In [ ]:
def electrode_likelihood_ratio_test(data, fband, base_vars, additional_vars, n_permutations=1000, 
                                   random_state=42, min_trials=10):
    """
    Perform permutation-corrected likelihood ratio test to assess whether adding 
    additional variables significantly improves model fit beyond a base model.
    
    Parameters:
    -----------
    data : pandas.DataFrame
        The chan_dfs_feedback dataframe containing TFR and behavioral data
    fband : str
        Frequency band to analyze (e.g., 'alpha', 'beta', 'hfa', 'high_theta', 'low_theta')
    base_vars : list
        List of variables in the base model (e.g., ['preward'])
    additional_vars : list
        List of variables to test for additional explanatory power (e.g., ['zrpe'])
    n_permutations : int, default=1000
        Number of permutations for z-scoring
    random_state : int, default=42
        Random seed for reproducibility
    min_trials : int, default=10
        Minimum number of trials required per electrode for analysis
        
    Returns:
    --------
    results_df : pandas.DataFrame
        DataFrame with likelihood ratio test results for each electrode
        
    Output Columns:
    ---------------
    - unique_elec_id: Electrode identifier
    - participant, region, channel, hemi: Electrode metadata
    - fband: Frequency band analyzed
    - n_trials: Number of trials used
    - base_vars: Variables in base model (comma-separated)
    - additional_vars: Variables tested for additional power (comma-separated)
    - base_r2: R² of base model
    - full_r2: R² of full model
    - lr_delta_r2: Difference in R² (full - base)
    - lr_z_score: Z-score of likelihood ratio against permutation distribution
    - lr_p_value: Uncorrected p-value (one-tailed)
    - lr_p_fdr: FDR-corrected p-value
    - {var}_beta: Beta coefficient for each additional variable
    
    Example:
    --------
    # Test if zrpe adds explanatory power beyond preward
    results = electrode_likelihood_ratio_test(
        data=chan_dfs_feedback,
        fband='hfa',
        base_vars=['preward'],
        additional_vars=['zrpe'],
        n_permutations=500,
        min_trials=15
    )
    
    # Filter for significant results
    sig_results = results[results['lr_p_fdr'] < 0.05]
    
    # Group by region to see which regions show improvement
    region_summary = results.groupby('region')[['lr_delta_r2', 'lr_z_score']].mean()
    """
    
    import numpy as np
    import pandas as pd
    from scipy import stats
    from sklearn.linear_model import LinearRegression
    import warnings
    warnings.filterwarnings('ignore')
    
    # Set random seed
    np.random.seed(random_state)
    
    # Filter data for specific frequency band
    data = data[data['fband'] == fband]
    
    if len(data) == 0:
        raise ValueError(f"No data found for frequency band '{fband}'")
    
    # Initialize results storage
    results = []
    
    # Get unique electrodes
    unique_electrodes = data['unique_elec_id'].unique()
    
    print(f"Analyzing {len(unique_electrodes)} electrodes for frequency band '{fband}'")
    print(f"Base model variables: {base_vars}")
    print(f"Additional variables: {additional_vars}")
    
    for i, elec_id in enumerate(unique_electrodes):
        if i % 50 == 0:  # Progress indicator
            print(f"Processing electrode {i+1}/{len(unique_electrodes)}")
            
        # Get data for this electrode
        elec_data = data[data['unique_elec_id'] == elec_id].copy()
        
        # Check minimum trial requirement
        if len(elec_data) < min_trials:
            continue
            
        # Prepare variables for regression
        y = elec_data['tfr'].values
        
        # Check for valid base variables
        valid_base_vars = []
        base_X_data = []
        
        for var in base_vars:
            if var in elec_data.columns:
                var_data = elec_data[var].values
                if np.sum(~np.isnan(var_data)) >= min_trials:
                    valid_base_vars.append(var)
                    base_X_data.append(var_data)
        
        # Check for valid additional variables
        valid_additional_vars = []
        additional_X_data = []
        
        for var in additional_vars:
            if var in elec_data.columns:
                var_data = elec_data[var].values
                if np.sum(~np.isnan(var_data)) >= min_trials:
                    valid_additional_vars.append(var)
                    additional_X_data.append(var_data)
        
        if len(valid_base_vars) == 0 or len(valid_additional_vars) == 0:
            continue
            
        # Stack variables
        base_X = np.column_stack(base_X_data)
        additional_X = np.column_stack(additional_X_data)
        full_X = np.column_stack([base_X, additional_X])
        
        # Remove rows with any NaN values
        valid_mask = ~(np.isnan(y) | np.any(np.isnan(base_X), axis=1) | np.any(np.isnan(additional_X), axis=1))
        y_clean = y[valid_mask]
        base_X_clean = base_X[valid_mask]
        full_X_clean = full_X[valid_mask]
        
        if len(y_clean) < min_trials:
            continue
            
        # Perform actual regressions
        try:
            # Base model
            base_reg = LinearRegression().fit(base_X_clean, y_clean)
            base_r2 = base_reg.score(base_X_clean, y_clean)
            
            # Full model
            full_reg = LinearRegression().fit(full_X_clean, y_clean)
            full_r2 = full_reg.score(full_X_clean, y_clean)
            
            # Calculate actual likelihood ratio (R² difference)
            actual_lr = full_r2 - base_r2
            
            # Permutation testing
            permuted_lrs = []
            
            for perm in range(n_permutations):
                # Shuffle the dependent variable (TFR)
                y_shuffled = np.random.permutation(y_clean)
                
                # Fit base model on shuffled data
                base_reg_perm = LinearRegression().fit(base_X_clean, y_shuffled)
                base_r2_perm = base_reg_perm.score(base_X_clean, y_shuffled)
                
                # Fit full model on shuffled data
                full_reg_perm = LinearRegression().fit(full_X_clean, y_shuffled)
                full_r2_perm = full_reg_perm.score(full_X_clean, y_shuffled)
                
                # Calculate permuted likelihood ratio
                permuted_lr = full_r2_perm - base_r2_perm
                permuted_lrs.append(permuted_lr)
            
            # Convert to array for easier computation
            permuted_lrs = np.array(permuted_lrs)
            
            # Compute z-score for likelihood ratio
            perm_mean = np.mean(permuted_lrs)
            perm_std = np.std(permuted_lrs)
            
            if perm_std > 0:
                lr_z_score = (actual_lr - perm_mean) / perm_std
                # One-tailed p-value (testing if additional variables improve model)
                lr_p_value = 1 - stats.norm.cdf(lr_z_score)
            else:
                lr_z_score = 0
                lr_p_value = 1
            
            # Store results
            electrode_info = elec_data.iloc[0]  # Get electrode metadata
            
            result_row = {
                'unique_elec_id': elec_id,
                'participant': electrode_info['participant'],
                'region': electrode_info['region'],
                'channel': electrode_info['channel'],
                'hemi': electrode_info['hemi'],
                'fband': fband,
                'n_trials': len(y_clean),
                'base_vars': ', '.join(valid_base_vars),
                'additional_vars': ', '.join(valid_additional_vars),
                'base_r2': base_r2,
                'full_r2': full_r2,
                'lr_delta_r2': actual_lr,
                'lr_z_score': lr_z_score,
                'lr_p_value': lr_p_value,
            }
            
            # Add beta coefficients for additional variables
            additional_betas = full_reg.coef_[len(valid_base_vars):]
            for j, var in enumerate(valid_additional_vars):
                result_row[f'{var}_beta'] = additional_betas[j]
            
            results.append(result_row)
            
        except Exception as e:
            print(f"Error processing electrode {elec_id}: {e}")
            continue
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # Add FDR correction for multiple comparisons
    if len(results_df) > 0:
        from statsmodels.stats.multitest import multipletests
        
        # FDR correction for likelihood ratio p-values
        valid_p_values = results_df['lr_p_value'].dropna()
        if len(valid_p_values) > 0:
            _, p_corrected, _, _ = multipletests(valid_p_values, method='fdr_bh')
            results_df['lr_p_fdr'] = np.nan
            results_df.loc[valid_p_values.index, 'lr_p_fdr'] = p_corrected
        else:
            # If no valid p-values, create empty column
            results_df['lr_p_fdr'] = np.nan
    else:
        # If no results at all, create empty DataFrame with expected columns
        results_df = pd.DataFrame(columns=[
            'unique_elec_id', 'participant', 'region', 'channel', 'hemi', 'fband', 
            'n_trials', 'base_vars', 'additional_vars', 'base_r2', 'full_r2', 
            'lr_delta_r2', 'lr_z_score', 'lr_p_value', 'lr_p_fdr'
        ])
    
    print(f"Analysis complete. Results for {len(results_df)} electrodes.")
    return results_df


# Example usage in your notebook:
"""
# Test if zrpe adds explanatory power beyond preward for HFA
lr_results = electrode_likelihood_ratio_test(
    data=chan_dfs_feedback,
    fband='hfa',
    base_vars=['preward'],
    additional_vars=['zrpe'],
    n_permutations=500,
    min_trials=15
)

# View significant results
sig_electrodes = lr_results[lr_results['lr_p_fdr'] < 0.05]
print(f"Number of electrodes where zrpe adds significant explanatory power: {len(sig_electrodes)}")

# Summarize by region
region_summary = lr_results.groupby('region').agg({
    'lr_delta_r2': 'mean',
    'lr_z_score': 'mean',
    'lr_p_fdr': lambda x: (x < 0.05).sum()
}).rename(columns={'lr_p_fdr': 'n_significant'})
print("\nRegion Summary:")
print(region_summary.sort_values('lr_delta_r2', ascending=False))

# Test multiple additional variables at once
lr_results_multi = electrode_likelihood_ratio_test(
    data=chan_dfs_feedback,
    fband='beta',
    base_vars=['preward'],
    additional_vars=['zrpe', 'zQ'],
    n_permutations=500
)
"""



In [ ]:
# Test if zrpe adds explanatory power beyond preward
lr_results = electrode_likelihood_ratio_test(
    data=chan_dfs_feedback,
    fband='gamma',
    base_vars=['preward'],
    additional_vars=['zrpe'],
    n_permutations=100,
    min_trials=15
)

# View significant results
sig_electrodes = lr_results[lr_results['lr_p_fdr'] < 0.05]
print(f"Significant electrodes: {len(sig_electrodes)}")

In [ ]:
# lr_results.to_csv(f'{base_dir}/scratch/MemoryBandit/results/electrode_lr_rpe_reward.csv', index=False)


In [ ]:
# Summarize by region
region_summary = lr_results.groupby('region')['lr_z_score'].mean().sort_values(ascending=False)
print(region_summary)

In [ ]:
# IS there a correlation between the RPE coding in AMY, dmPFC, or OFC and RL parameters?

# 1. merge lr_results with learn_df on 'participant'
merged_df = lr_results.merge(learn_df, on='participant', how='left')

# 2 run a mixedlm with the following formula: 'alpha ~ lr_z_score + (1|participant)'
mixedlm = smf.mixedlm('lr_z_score ~ 1 + alpha', 
groups='participant',
data=merged_df[merged_df.region.isin(['AMY'])])

# 3. run the model
model = mixedlm.fit()

# 4. print the summary
print(model.summary())

In [ ]:
sns.regplot(data=merged_df[merged_df.region.isin(['AMY', 'OFC'])], 
x='alpha', y='lr_z_score', scatter=True)

## Now repeat for memory:

In [ ]:
# LOAD the data
analysis_evs = ['feedback_start']
chan_dfs = [] 
glms = []
elec_counter = 0 
block = [0.0, 1.5]
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')

            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            # chan_df['rpe_cat'] = pd.qcut(chan_df.rpe, 3, labels=['neg', 'neutral', 'pos'])
            chan_df['urpe'] = np.abs(chan_df.rpe)
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
            chan_df['preward'] = chan_df.apply(lambda x: 0.5 if x.reward==1 else -0.5 if x.reward==0 else np.nan, axis=1)

            # # # learning only: 
            chan_df = chan_df[chan_df.gamble_rt>0.3]
            chan_df = chan_df[chan_df.gamble_rt<10]

            # # with memory
            memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits', 'recog_rt', 'zrpe', 'zQ', 'zRT_rpe', 'preward', 'ztrials_since_reversal', 'zprior_rpe']]
            memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
            memory_df['phit'] = memory_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)
            chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])

            chan_df = chan_df[chan_df.recog_rt>0.3]
            chan_df = chan_df[chan_df.recog_rt<10]

            elec_counter +=1

            chan_dfs.append(chan_df)

chan_dfs_feedback = pd.concat(chan_dfs)
chan_dfs_feedback['zAge'] =(chan_dfs_feedback.age - np.nanmean(chan_dfs_feedback.age)) / (2*np.nanstd(chan_dfs_feedback.age))

rois = ['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC', 'dlPFC']
# remove rows where region is not in the list   
chan_dfs_feedback = chan_dfs_feedback[chan_dfs_feedback.region.isin(rois)]

In [ ]:
# Example usage of the regression function

# Define frequency bands available in the data
available_bands = chan_dfs_feedback['fband'].unique()
print("Available frequency bands:", available_bands)

# Example 1: Simple regression with RPE
print("\n=== Example 1: TFR ~ RPE (z-scored) ===")
results_rpe = electrode_regression_analysis(
    data=chan_dfs_feedback,
    fband='hfa',  # Choose frequency band
    independent_vars=['phit'],  # RPE z-scored
    n_permutations=500,  # Fewer permutations for faster testing
    min_trials=15
)

print(f"Results shape: {results_rpe.shape}")
print(f"Columns: {list(results_rpe.columns)}")
print(f"\nTop 5 electrodes by R² z-score:")
# print(results_rpe.nlargest(5, 'r2_z_score')[['region', 'channel', 'actual_r2', 'r2_z_score', 'zrpe_beta', 'zrpe_z_score']])


In [ ]:
results_rpe['abs_phit_beta'] = np.abs(results_rpe['phit_z_score'])

In [ ]:
results_rpe.groupby(['region']).mean(numeric_only=True).sort_values(by='phit_z_score', ascending=False)

Now we are looking at memory encoding itself, and we see that mostly HPC and AMY increase their activity during successful memory encoding.

Does accounting for perceptual memorability add anything? These might be more perceptual regions.


In [ ]:
# # Test if zrpe adds explanatory power beyond preward
# lr_results = electrode_likelihood_ratio_test(
#     data=chan_dfs_feedback,
#     fband='hfa',
#     base_vars=['phit'],
#     additional_vars=['zpm'],
#     n_permutations=500,
#     min_trials=15
# )

# # View significant results
# sig_electrodes = lr_results[lr_results['lr_p_fdr'] < 0.05]
# print(f"Significant electrodes: {len(sig_electrodes)}")

In [ ]:
# # Summarize by region
# region_summary = lr_results.groupby('region')['lr_z_score'].mean().sort_values(ascending=False)
# print(region_summary)

According to this (which I won't read into too much), ACC and dlPFC seem to integrate information related to perception in addition to memory encoding processes. This is interesting but not that interesting, and I will ignore it for the most part.